### Data Ingestion

In [4]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

print("Environment variables loaded successfully")

Environment variables loaded successfully


In [5]:
# src/extractors/base.py

"""
Base extractor class and file type detection for document ingestion.
Provides a unified interface for all document extractors.
"""

import magic
from pathlib import Path
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Optional, Dict, Any, List


@dataclass
class ExtractedDocument:
    """
    Represents a document after content extraction.
    Contains the raw text and associated metadata.
    """
    content: str                          # The extracted text content
    metadata: Dict[str, Any]              # Document metadata (title, author, etc.)
    source_path: str                      # Original file path
    file_type: str                        # Detected MIME type
    extraction_method: str                # Which extractor was used
    page_count: Optional[int] = None      # Number of pages (if applicable)
    word_count: Optional[int] = None      # Total word count


class BaseExtractor(ABC):
    """
    Abstract base class for document extractors.
    All file-type-specific extractors inherit from this class.
    """

    # List of MIME types this extractor can handle
    supported_mimetypes: List[str] = []

    @abstractmethod
    def extract(self, file_path: Path) -> ExtractedDocument:
        """
        Extract text content from a document.

        Args:a
            file_path: Path to the document file

        Returns:
            ExtractedDocument containing the extracted content and metadata
        """
        pass

    @abstractmethod
    def can_handle(self, mimetype: str) -> bool:
        """
        Check if this extractor can handle the given MIME type.

        Args:
            mimetype: The MIME type to check

        Returns:
            True if this extractor can process the file type
        """
        pass


class FileTypeDetector:
    """
    Detects file types using both magic bytes and file extensions.
    Magic bytes are more reliable than extensions for security.
    """

    def __init__(self):
        # Initialize libmagic for MIME type detection
        self.magic = magic.Magic(mime=True)

        # Fallback extension mappings for edge cases
        self.extension_map = {
            '.pdf': 'application/pdf',
            '.docx': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
            '.doc': 'application/msword',
            '.txt': 'text/plain',
            '.md': 'text/markdown',
            '.html': 'text/html',
            '.htm': 'text/html',
            '.json': 'application/json',
            '.csv': 'text/csv',
            '.xlsx': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
            '.pptx': 'application/vnd.openxmlformats-officedocument.presentationml.presentation',
        }

    def detect(self, file_path: Path) -> str:
        """
        Detect the MIME type of a file.
        Uses magic bytes first, falls back to extension if needed.

        Args:
            file_path: Path to the file

        Returns:
            MIME type string (e.g., 'application/pdf')
        """
        try:
            # Read first 2048 bytes for magic number detection
            with open(file_path, 'rb') as f:
                header = f.read(2048)

            # Detect using magic bytes
            mimetype = self.magic.from_buffer(header)

            # Handle generic types by checking extension
            if mimetype in ('application/octet-stream', 'text/plain'):
                ext = file_path.suffix.lower()
                if ext in self.extension_map:
                    return self.extension_map[ext]

            return mimetype

        except Exception as e:
            # Fallback to extension-based detection
            ext = file_path.suffix.lower()
            return self.extension_map.get(ext, 'application/octet-stream')


class ExtractorRegistry:
    """
    Registry that maps file types to their extractors.
    Automatically routes documents to the correct extractor.
    """

    def __init__(self):
        self.extractors: List[BaseExtractor] = []
        self.detector = FileTypeDetector()

    def register(self, extractor: BaseExtractor) -> None:
        """
        Register an extractor with the registry.

        Args:
            extractor: The extractor instance to register
        """
        self.extractors.append(extractor)

    def get_extractor(self, file_path: Path) -> Optional[BaseExtractor]:
        """
        Get the appropriate extractor for a file.

        Args:
            file_path: Path to the file

        Returns:
            The extractor that can handle this file type, or None
        """
        mimetype = self.detector.detect(file_path)

        for extractor in self.extractors:
            if extractor.can_handle(mimetype):
                return extractor

        return None

In [6]:
import os
import re
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
print(f"Data directory: {data_dir}")
all_documents = []


def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"(?m)^\s*(page\s*\d+|\d+\s*of\s*\d+)\s*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Load each PDF in the directory
for pdf_file in os.listdir(data_dir):
    if pdf_file.endswith('.pdf'):
        file_path = os.path.join(data_dir, pdf_file)
        loader = PyMuPDFLoader(file_path)
        docs = loader.load()
        for doc in docs:
            doc.page_content = clean_text(doc.page_content)
        all_documents.extend(docs)

print(f"Loaded {len(all_documents)} documents")
all_documents

Data directory: c:\Rag ai agent chat bot\data
Loaded 20 documents


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-06-18T14:59:12+05:30', 'source': 'c:\\Rag ai agent chat bot\\data\\AI Training Document.pdf', 'file_path': 'c:\\Rag ai agent chat bot\\data\\AI Training Document.pdf', 'total_pages': 20, 'format': 'PDF 1.7', 'title': '', 'author': 'Shivani Gupta', 'subject': '', 'keywords': '', 'moddate': '2025-06-18T14:59:12+05:30', 'trapped': '', 'modDate': "D:20250618145912+05'30'", 'creationDate': "D:20250618145912+05'30'", 'page': 0}, page_content='User Agreement 1. Introduction This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms posted on and in our sites, applications, tools, and services (collectively "Services") set out the terms on which eBay offers you access to and use of our Services. You can find an overview of our policies here. The Mobile Application Terms of Use, all policies, and additional terms posted on and in our Services are 

### Chunks Creation

In [8]:
def chunk_documents(documents, chunk_size=300, chunk_overlap=80):
    """
    Splits documents into sentence-based chunks with overlap.
    """
    import os
    import sys
    project_root = os.path.dirname(os.getcwd())
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

    from src.rag.chunking import sentence_chunk_documents
    return sentence_chunk_documents(documents, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

# Use the function
chunks = chunk_documents(all_documents)
print(f"Created {len(chunks)} chunks from {len(all_documents)} documents")
chunks

Created 432 chunks from 20 documents


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-06-18T14:59:12+05:30', 'source': 'c:\\Rag ai agent chat bot\\data\\AI Training Document.pdf', 'file_path': 'c:\\Rag ai agent chat bot\\data\\AI Training Document.pdf', 'total_pages': 20, 'format': 'PDF 1.7', 'title': '', 'author': 'Shivani Gupta', 'subject': '', 'keywords': '', 'moddate': '2025-06-18T14:59:12+05:30', 'trapped': '', 'modDate': "D:20250618145912+05'30'", 'creationDate': "D:20250618145912+05'30'", 'page': 0, 'chunk_index': 1, 'chunking': 'sentence'}, page_content='User Agreement 1. Introduction This User Agreement, the Mobile Application Terms of Use, and all policies and additional terms posted on and in our sites, applications, tools, and services (collectively "Services") set out the terms on which eBay offers you access to and use of our Services.'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '

### Embeddings

In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Save FAISS index to project root so app can find it
# Navigate up from notebooks folder to project root
project_root = os.path.dirname(os.getcwd())
FAISS_INDEX_DIR = os.path.join(project_root, "vectordb", "faiss_index")

def create_vector_store(chunks):
    """
    Creates a FAISS vector store with HuggingFace embeddings.
    Saves the index to disk so it persists across sessions.

    Args:
        chunks: List of chunked Document objects

    Returns:
        FAISS vector store with embedded documents
    """
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-MiniLM-L3-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(FAISS_INDEX_DIR)
    print(f"Vector store saved to: {FAISS_INDEX_DIR}")
    return vectorstore

print(f"Number of chunks: {len(chunks)}")
if len(chunks) == 0:
    print("No chunks found! Check if PDFs were loaded in cell 2.")
else:
    vectorstore = create_vector_store(chunks)
    print(f"Vector store created - total stored: {vectorstore.index.ntotal}")

### Retrival Layer

In [ ]:
# Add project root to path so we can import src modules
import sys
import os
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import retriever from src module (single source of truth)
from src.rag.retriever import RagRetriever, load_retriever

# --- Quick smoke-test ---
retriever = RagRetriever(vector_store=vectorstore)
results = retriever.retrieve("What is this document about?", top_k=3)
for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} (score={r['score']}) ---")
    print(r["content"][:300])

In [ ]:
# Add project root to path so we can import src modules
import sys
import os
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Generation layer - Import from src modules (single source of truth)
from src.llm.ollama_client import load_ollama_client
from src.rag.pipeline import ask_rag

# Load local Ollama client
mistral_client = load_ollama_client()
print("Local Ollama model ready.")

# Auto-restore retriever if kernel was restarted and previous cells were not re-run.
if "retriever" not in globals():
    from src.rag.retriever import load_retriever
    retriever = load_retriever()
    print("Retriever initialized from vectorstore.")


# ---- Demo run ----
question = "Summarize the key topics discussed in these PDFs."
result = ask_rag(question, retriever=retriever, mistral_client=mistral_client)

print("Answer:\n")
print(result["answer"])
print(f"\nSources ({result['retrieved_count']} chunks used):")
for idx, src in enumerate(result["sources"], 1):
    print(f"  {idx}. {src['source']} (page={src['page']}, score={src['score']})")